In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using {device}")

tensor_size = (1000, 1000)
a = torch.randn(tensor_size , device = device) #random value but with normal distribution(mean = 0 , std = 1)
b = torch.randn(tensor_size , device = device) #random vlaue but with normal distribution(mean - 0 , std = 1)
c = a + b
print("result shape(Moved to CPU for printing)" , c.cpu().shape) #GPU printing is slower

print("Current GPU memory Usage:")
print(f"Allocated:{torch.cuda.memory_allocated(device)/1024**2: .2f} MB") #from bytes to MB( byte to KB -> KB to MB - > 1024x1024)
print(f"Cached: {torch.cuda.memory_reserved(device)/1024**2 :.2f} MB")
torch.cuda.empty_cache()
print("After clearing cache:")
print(f"Allocated:{torch.cuda.memory_allocated(device)/1024**2: .2f} MB")

Using cuda
result shape(Moved to CPU for printing) torch.Size([1000, 1000])
Current GPU memory Usage:
Allocated: 11.44 MB
Cached: 20.00 MB
After clearing cache:
Allocated: 11.44 MB


### Building and Training Neural Networks with PyTorch
Step 1: Define the Neural Network Class
In this step, we’ll define a class that inherits from torch.nn.Module. We’ll create a simple neural network with an input layer, a hidden layer, and an output layer.

In [ ]:
import torch
import torch.nn as nn

class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(2, 4)
        self.fc2 = nn.Linear(4, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

###Prepare the data

In [ ]:
X_train = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_train = torch.tensor([[0.0], [1.0], [1.0], [0.0]])


### Prepare the Model , loss function , optimizer

In [ ]:
model = SimpleNN()
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)##model.parameter() gives access to all the parameters(weights , biases) of the model and lr is learning rate set to .1

### Step 5: Training the Model

In [ ]:
for epoch in range(100):
    model.train()

    # Forward pass
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    # Backward pass and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.2229
Epoch [20/100], Loss: 0.2076
Epoch [30/100], Loss: 0.1941
Epoch [40/100], Loss: 0.1797
Epoch [50/100], Loss: 0.1648
Epoch [60/100], Loss: 0.1490
Epoch [70/100], Loss: 0.1338
Epoch [80/100], Loss: 0.1181
Epoch [90/100], Loss: 0.1029
Epoch [100/100], Loss: 0.0870


### Model Evaluation

In [ ]:
model.eval()
with torch.no_grad():
  test_data = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
  predictions = model(test_data)
  print(f"Predictions :\n {predictions}")


Predictions :
 tensor([[0.3158],
        [0.5913],
        [0.8556],
        [0.2321]])


### Optimizing Model Training with PyTorch Datasets
1. Efficient Data Handling with Datasets and DataLoaders
- Dataset and DataLoader enables Batch processing .
  - What is Batch Processing?-
Batch processing means instead of feeding one sample at a time, you feed multiple samples (a "batch") to the model.


**Note:**<br>
ask gpt to generate example based on this -<br>
Single-sample training (slow, underutilizes GPU)<br>
Batch training (faster, optimized for GPU)<br>



In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self):
        self.data = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
        self.labels = torch.tensor([0, 1, 0])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

dataset = MyDataset()
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch in dataloader:
    print("Batch Data:", batch[0])
    print("Batch Labels:", batch[1])

Batch Data: tensor([[5., 6.],
        [1., 2.]])
Batch Labels: tensor([0, 0])
Batch Data: tensor([[3., 4.]])
Batch Labels: tensor([1])


### Batch Processing for GPU Utilization

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset , DataLoader
import time

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [ ]:
X = torch.randn(1000,10)
y = torch.randn(1000 , 1)

In [ ]:
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset , batch_size = 100, shuffle = True)


In [ ]:
##Simple NN
model = nn.Sequential(
    nn.Linear(10, 50),
    nn.ReLU(),
    nn.Linear(50, 1)
).to(device)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters() , lr=0.1)

In [ ]:
#Batch training
start = time.time()
model.train()

for epoch in range(5):
    for batch_X, batch_y in dataloader:
        batch_X , batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        ##forward Pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        #backward pass and optimize
        loss.backward()
        optimizer.step()
end = time.time()
print(f"total time(Batch processing utilizing GPU: {end - start:.4f})")

total time(Batch processing utilizing GPU: 0.1327)


`So if you use Batch processing and CPU , there will be utilization but If you do use Batch processing + GPU , then you're fully utilizing the GPU `